# Claude API Learning Journey

## Step 1 — First API Request
Install dependencies, load API key from `.env`, initialise the Anthropic client, and make the first `client.messages.create()` call.

## Step 2 — Multi-Turn Conversations
Claude is stateless — maintain a `messages` list and send the full history with every request. Helper functions: `add_user_message()`, `add_assistant_message()`, `chat()`.

## Step 3 — Chat Exercise: Continuous Chat Loop
`while True` loop that accepts user input, sends the full conversation history to Claude, appends the response, and repeats — creating a stateful terminal chat session.

## Step 4 — System Prompts
Shape Claude's role and behaviour with a `system` parameter. Updated `chat()` to accept an optional `system` arg (conditionally included — API rejects `system=None`).

## Step 5 — System Prompts Exercise
Applied a one-sentence system prompt `"You are a Python engineer who writes very concise code"` — Claude returned a 1-line solution vs. a verbose multi-function response without the prompt.

## Step 6 — Temperature
`temperature` (0.0–1.0) controls token selection randomness. Low → deterministic/factual. High → creative/varied. Added as an optional param to `chat()` defaulting to `1.0`. Verified with movie idea generation — temperature 0 produced near-identical responses across 3 runs, temperature 1 produced distinct ideas each time.

## Step 7 — Response Streaming
Stream Claude's response word by word using `stream=True` or `client.messages.stream()`. Three patterns: raw events, simplified `text_stream`, and `get_final_message()` for the assembled result after streaming.

## Step 8 — Structured Data
Force clean structured output (JSON, code, CSV) without markdown wrappers or explanatory prose. Course technique: assistant message prefill + stop sequences — not supported on Claude 4. Workaround: system prompt instructing raw output directly.

## Step 9 — Structured Data Exercise
Applied the system prompt workaround to generate raw AWS CLI bash commands. Course prefill approach (` ```bash ` + stop sequence) fails on Claude 4 — replaced with `system="Output raw bash commands only..."` for compatible clean output.

## Step 10 — Generating Test Datasets
Auto-generate eval datasets using Claude itself. Course technique uses assistant prefill + stop sequences for raw JSON — not supported on Claude 4. Fixed with system prompt workaround. Used `json.dumps(dataset, indent=2)` for readable output. Dataset saved to `dataset.json` for reuse across eval runs.

In [23]:
# Install Dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [24]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [25]:
# Create an API client
from anthropic import Anthropic 
client = Anthropic()
model = "claude-sonnet-4-6"

In [64]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
        
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [82]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects. Return a raw JSON array only — no markdown, no explanation, no code fences.
"""
    messages = []
    add_user_message(messages, prompt)
    # Claude 4 doesn't support assistant prefill — enforce raw JSON via system prompt instead
    text = chat(
        messages,
        system="Output raw JSON only. No markdown, no explanation, no code fences."
    )
    return json.loads(text)

dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[
  {
    "id": 1,
    "type": "Python",
    "task_description": "Write a Python function that takes an S3 bucket name and a prefix as arguments, and returns a list of all object keys in that bucket matching the given prefix using the boto3 library.",
    "expected_output_description": "A Python function using boto3's S3 paginator to list all object keys under a given prefix and return them as a list of strings.",
    "example_input": {
      "bucket_name": "my-data-bucket",
      "prefix": "logs/2024/"
    },
    "example_output": [
      "logs/2024/jan.log",
      "logs/2024/feb.log",
      "logs/2024/mar.log"
    ]
  },
  {
    "id": 2,
    "type": "JSON",
    "task_description": "Write an AWS IAM policy JSON document that allows a user to only list and get objects from a specific S3 bucket named 'finance-reports', but explicitly denies the ability to delete any objects.",
    "expected_output_description": "A valid IAM policy JSON object with two statements: one Allow statement for